# Qwen3.5 Test Notebook

Dieses Notebook ist ein **qwen35_stable-only** MWE fuer lokale Tests mit `Qwen/Qwen3.5-0.8B`.

## Ziel

Das Notebook zeigt drei Dinge:

- `transformers` auf CPU funktioniert fuer denselben Labeling-Prompt.
- `GGUF + llama-server` funktioniert lokal und ist fuer diesen Test deutlich schneller.
- BERTopic kann fuer das Topic-Labelling denselben `llama-server` nutzen.

## Entscheidungsstand

- `ADS_env` ist fuer dieses Notebook **out of scope**.
- Offizielles BERTopic `representation.LlamaCPP` ist hier bewusst **nicht** Teil der MWE.
- Grund: `llama-cpp-python` laesst sich in `qwen35_stable` per `uv/pip` aktuell nicht sauber nutzen, weil der Installationspfad auf einen lokalen Windows-Build faellt und dort die Build-Toolchain fehlt.
- `ADS_env` ist dafuer kein Gegenbeispiel: Dort kommt `llama-cpp-python` bereits als vorgebautes `conda-forge`-Paket.
- Selbst mit installiertem `llama-cpp-python` ist der offizielle BERTopic-`LlamaCPP`-Pfad fuer dieses `Qwen3.5`-GGUF lokal bereits als nicht belastbar etabliert.
- Der funktionierende und schnelle Pfad fuer dieses Notebook ist deshalb `llama-server`, auch fuer BERTopic-Labelling.

## Nicht Ziel dieses Notebooks

- keine Env-Reparatur
- keine `llama_cpp`-Experimente
- keine alternativen BERTopic-Backends
- keine Zusatzlogik fuer Install-Fallbacks oder Workarounds


In [3]:
import json
import os
import statistics
import subprocess
import time
import urllib.request
import warnings
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore", message="IProgress not found.*")

import torch
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3.5-0.8B"
CPU_THREADS = 4
LABEL_KWARGS = {"max_new_tokens": 8, "temperature": 0.0}
SAMPLE_KEYWORDS = "telescope, orbit, spacecraft, galaxy, mission"
SAMPLE_DOCS = "observations of distant galaxies, orbital dynamics, and deep-space missions"

PROJECT_ROOT = Path.cwd()
CACHE_DIR = PROJECT_ROOT / "data" / "cache"
EMBED_CACHE_DIR = CACHE_DIR / "embeddings"

torch.set_num_threads(CPU_THREADS)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_messages(keywords: str, representative_docs: str) -> list[dict]:
    return [
        {
            "role": "system",
            "content": "You label research topics with 1 to 3 words only. Output only the final label.",
        },
        {
            "role": "user",
            "content": (
                "Label this topic in 1-3 words only. "
                f"Topic keywords: {keywords}. "
                f"Representative docs: {representative_docs}."
            ),
        },
    ]

def render_prompt(keywords: str, representative_docs: str) -> str:
    return tokenizer.apply_chat_template(
        build_messages(keywords, representative_docs),
        tokenize=False,
        add_generation_prompt=True,
    )

def strip_think_tags(text: str) -> str:
    return text.replace("<think>", "").split("</think>", 1)[-1].strip()


In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Load publications from pipeline cache (359 docs, JSONL)
_pubs = []
with open(CACHE_DIR / "publications_tokenized.json", encoding="utf-8") as _f:
    for _line in _f:
        _line = _line.strip()
        if _line:
            _pubs.append(json.loads(_line))

docs = [p["full_text"] for p in _pubs]
print(f"Loaded {len(docs)} publications")

# Compute fresh embeddings – all-MiniLM-L6-v2 (locally cached, 384D, no save)
_embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
real_embeddings = _embed_model.encode(docs, show_progress_bar=True, convert_to_numpy=True)
print(f"Embeddings: {real_embeddings.shape}")


Loaded 359 publications


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 12/12 [00:08<00:00,  1.41it/s]

Embeddings: (359, 384)


In [5]:
import contextlib
import io

@contextlib.contextmanager
def silence_fds():
    saved_stdout = os.dup(1)
    saved_stderr = os.dup(2)
    with open(os.devnull, "w") as devnull:
        try:
            os.dup2(devnull.fileno(), 1)
            os.dup2(devnull.fileno(), 2)
            yield
        finally:
            os.dup2(saved_stdout, 1)
            os.dup2(saved_stderr, 2)
            os.close(saved_stdout)
            os.close(saved_stderr)

with silence_fds(), contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        device_map="cpu",
    )
model.eval()
model.generation_config.pad_token_id = tokenizer.pad_token_id

def label_topic_transformers(
    keywords: str,
    representative_docs: str,
    max_new_tokens: int = 8,
    temperature: float = 0.0,
) -> dict:
    inputs = tokenizer(
        [render_prompt(keywords, representative_docs)],
        add_special_tokens=False,
        return_tensors="pt",
    )
    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": temperature > 0.0,
        "pad_token_id": tokenizer.pad_token_id,
    }
    if temperature > 0.0:
        generation_kwargs["temperature"] = temperature

    t0 = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(**inputs, **generation_kwargs)
    t1 = time.perf_counter()

    generated_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], outputs)
    ]
    label = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return {
        "backend": "transformers",
        "label": strip_think_tags(label),
        "input_tokens": int(inputs["input_ids"].shape[-1]),
        "generate_s": round(t1 - t0, 2),
    }

with silence_fds(), contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    hf_sample = label_topic_transformers(SAMPLE_KEYWORDS, SAMPLE_DOCS, **LABEL_KWARGS)

print("backend:", hf_sample["backend"])
print("model_id:", MODEL_ID)
print("cpu_threads:", CPU_THREADS)
print("label:", hf_sample["label"])
print("input_tokens:", hf_sample["input_tokens"])
print("generate_s:", hf_sample["generate_s"])


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


backend: transformers
model_id: Qwen/Qwen3.5-0.8B
cpu_threads: 4
label: spacecraft
input_tokens: 76
generate_s: 10.67


In [6]:
import atexit
import shutil

resolved_llama_server = shutil.which("llama-server")
if resolved_llama_server is None:
    raise FileNotFoundError("llama-server not found on PATH")

LLAMA_SERVER_EXE = Path(resolved_llama_server)
GGUF_MODEL_PATH = Path(r"data\models\qwen35_gguf\Qwen_Qwen3.5-0.8B-Q4_K_M.gguf")
LLAMA_PORT = 8001

if not GGUF_MODEL_PATH.exists():
    raise FileNotFoundError(f"GGUF model not found: {GGUF_MODEL_PATH}")

llama_server_process = globals().get("llama_server_process")

def _post_json(url: str, payload: dict, timeout: float = 120.0) -> dict:
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))

def llama_server_ready(port: int = LLAMA_PORT) -> bool:
    try:
        _post_json(
            f"http://127.0.0.1:{port}/completion",
            {"prompt": "ping", "n_predict": 1, "temperature": 0.0},
            timeout=5.0,
        )
        return True
    except Exception:
        return False

def ensure_llama_server(
    port: int = LLAMA_PORT,
    cpu_threads: int = CPU_THREADS,
    timeout_s: float = 120.0,
) -> None:
    global llama_server_process

    if llama_server_ready(port):
        return

    llama_server_process = subprocess.Popen(
        [
            str(LLAMA_SERVER_EXE),
            "-m",
            str(GGUF_MODEL_PATH),
            "--host",
            "127.0.0.1",
            "--port",
            str(port),
            "--threads",
            str(cpu_threads),
            "--ctx-size",
            "4096",
            "--reasoning",
            "off",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        creationflags=getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0),
    )

    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if llama_server_process.poll() is not None:
            raise RuntimeError("llama-server exited before becoming ready")
        if llama_server_ready(port):
            return
        time.sleep(1.0)

    raise TimeoutError("llama-server did not become ready in time")

def stop_llama_server() -> None:
    global llama_server_process

    if llama_server_process is None:
        return
    if llama_server_process.poll() is None:
        llama_server_process.terminate()
        try:
            llama_server_process.wait(timeout=10.0)
        except subprocess.TimeoutExpired:
            llama_server_process.kill()
            llama_server_process.wait(timeout=10.0)
    llama_server_process = None

atexit.register(stop_llama_server)

def label_topic_llama_server(
    keywords: str,
    representative_docs: str,
    max_new_tokens: int = 8,
    temperature: float = 0.0,
    port: int = LLAMA_PORT,
) -> dict:
    ensure_llama_server(port=port)

    t0 = time.perf_counter()
    response = _post_json(
        f"http://127.0.0.1:{port}/completion",
        {
            "prompt": render_prompt(keywords, representative_docs),
            "n_predict": max_new_tokens,
            "temperature": temperature,
        },
    )
    t1 = time.perf_counter()

    return {
        "backend": "llama_server",
        "label": strip_think_tags(response.get("content", "")),
        "input_tokens": int(response.get("tokens_evaluated", -1)),
        "generate_s": round(t1 - t0, 2),
    }

print("llama-server exe:", LLAMA_SERVER_EXE)
print("gguf_model_path:", GGUF_MODEL_PATH)
print("llama_port:", LLAMA_PORT)

try:
    llama_sample = label_topic_llama_server(SAMPLE_KEYWORDS, SAMPLE_DOCS, **LABEL_KWARGS)
finally:
    stop_llama_server()

print("backend:", llama_sample["backend"])
print("port:", LLAMA_PORT)
print("label:", llama_sample["label"])
print("input_tokens:", llama_sample["input_tokens"])
print("generate_s:", llama_sample["generate_s"])


llama-server exe: C:\Users\rapha\AppData\Local\Microsoft\WinGet\Packages\ggml.llamacpp_Microsoft.Winget.Source_8wekyb3d8bbwe\llama-server.EXE
gguf_model_path: data\models\qwen35_gguf\Qwen_Qwen3.5-0.8B-Q4_K_M.gguf
llama_port: 8001
backend: llama_server
port: 8001
label: spacecraft
input_tokens: 76
generate_s: 0.58


In [7]:
# BENCHMARK_REPEATS = 5

# def benchmark_backend(
#     label_fn,
#     backend_name: str,
#     repeats: int = BENCHMARK_REPEATS,
#     warmup: bool = False,
# ) -> dict:
#     if warmup:
#         label_fn(SAMPLE_KEYWORDS, SAMPLE_DOCS, **LABEL_KWARGS)

#     runs = []
#     last_result = None
#     for _ in range(repeats):
#         last_result = label_fn(SAMPLE_KEYWORDS, SAMPLE_DOCS, **LABEL_KWARGS)
#         runs.append(last_result["generate_s"])

#     return {
#         "backend": backend_name,
#         "label": last_result["label"],
#         "input_tokens": last_result["input_tokens"],
#         "best_s": round(min(runs), 2),
#         "mean_s": round(statistics.mean(runs), 2),
#         "runs_s": runs,
#     }

# results = [benchmark_backend(label_topic_transformers, "transformers", warmup=True)]

# try:
#     results.append(benchmark_backend(label_topic_llama_server, "llama_server", warmup=True))
# finally:
#     stop_llama_server()

# print()
# print("Benchmark warm runs")
# print("-------------------")
# for result in results:
#     print(f"backend: {result['backend']}")
#     print(f"label: {result['label']}")
#     print(f"input_tokens: {result['input_tokens']}")
#     print(f"best_s: {result['best_s']}")
#     print(f"mean_s: {result['mean_s']}")
#     print(f"runs_s: {result['runs_s']}")
#     print()

# delta = round(results[0]["mean_s"] - results[1]["mean_s"], 2)
# print(f"mean_s delta (transformers - llama_server): {delta}")
# print("llama_server notebook process stopped:", llama_server_process is None)


In [8]:
import os
import warnings

import litellm
litellm.suppress_debug_info = True
os.environ.setdefault("LITELLM_LOG", "ERROR")

warnings.filterwarnings("ignore", category=SyntaxWarning, module=r"hdbscan")

from bertopic import BERTopic
from bertopic.representation import LiteLLM
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

BERTOPIC_PROMPT = """
I have a topic that contains the following keywords: [KEYWORDS]
The topic is best described by the following documents:
[DOCUMENTS]
Label this topic in 1-3 words only. Output only the label, nothing else.
"""

representation = LiteLLM(
    model="openai/Qwen3.5-0.8B",
    prompt=BERTOPIC_PROMPT,
    generator_kwargs={
        "max_tokens": 16,
        "temperature": 0.0,
        "api_base": f"http://127.0.0.1:{LLAMA_PORT}/v1",
        "api_key": "placeholder",
    },
    nr_docs=3,
)

topic_model = BERTopic(
    embedding_model=None,
    representation_model=representation,
    vectorizer_model=CountVectorizer(stop_words="english"),
    umap_model=UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42,
        low_memory=True,
    ),
    hdbscan_model=HDBSCAN(
        min_cluster_size=5,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=False,
    ),
    min_topic_size=5,
    calculate_probabilities=False,
    verbose=False,
)

try:
    ensure_llama_server()
    topics, _ = topic_model.fit_transform(docs, embeddings=real_embeddings)
finally:
    stop_llama_server()

n_topics = len(set(t for t in topics if t != -1))
n_outliers = sum(1 for t in topics if t == -1)
print(f"backend: BERTopic + LiteLLM → llama-server")
print(f"n_docs: {len(docs)}, n_topics: {n_topics}, n_outliers: {n_outliers}")
print()
print("generated labels")
print("----------------")
for topic_id in sorted({t for t in topics if t != -1}):
    print(f"topic_id={topic_id:2d}: {topic_model.get_topic(topic_id)[0][0]}")
print("llama_server stopped:", llama_server_process is None)


backend: BERTopic + LiteLLM → llama-server
n_docs: 359, n_topics: 24, n_outliers: 82

generated labels
----------------
topic_id= 0: inflationary model
topic_id= 1: Causally continuous spacetimes
topic_id= 2: quantum cosmology
topic_id= 3: early universe
topic_id= 4: singularities
topic_id= 5: Quantum Universe
topic_id= 6: CFT
topic_id= 7: supersymmetry
topic_id= 8: wormholes
topic_id= 9: quantum coherence loss
topic_id=10: path integral regularization
topic_id=11: black holes
topic_id=12: universe
topic_id=13: expanding universe
topic_id=14: General Relativity
topic_id=15: gravitation
topic_id=16: cosmological constant
topic_id=17: General Relativity
topic_id=18: black hole
topic_id=19: cosmic strings
topic_id=20: arrow time direction thermodynamic perturbations
topic_id=21: Black Holes
topic_id=22: quantum gravity
topic_id=23: Holographic Cosmology
llama_server stopped: True
